# Trade & Price: UN Comtrade Crude Oil Trade Data (HS 2709), 2019-2025

## Deliverable 1 – Task 1:

This notebook retrieves annual crude oil trade data from UN Comtrade for 2019-2025, including both exporter-reported (X) and importer-reported (M) trade flows.

Data source: [UN Comtrade API](https://comtradeapi.un.org/)

The official documentation for comtradeapicall is available at: [comtradeapicall](https://pypi.org/project/comtradeapicall/)

### A. API Setup

Import the required Python packages, load the API subscription key from .env, and define common query parameters for crude oil (HS 2709).

In [3]:
# Import standard python libraries
import pandas as pd
import numpy as np

In [4]:
# Run initially to install package from PyPi
# %pip install comtradeapicall

# Import comtradeapicall package
import comtradeapicall

# Get subscription key from .env
import os
from dotenv import load_dotenv

load_dotenv()

subscription_key = os.getenv("SUBSCRIPTION_KEY")

### B. Retrieve and Review Comtrade Data
Two transport-mode:

- All modes: No transport-mode restriction (motCode=None).
- Sea only: Maritime transport records (motCode=2100).

Both datasets include importer- and exporter-reported flows. Preview the returned records and inspect the available fields.

In [6]:
# Preview the data
comtrade_preview = comtradeapicall.previewFinalData(typeCode='C', freqCode='M', clCode='HS', period='202205',
                                        reporterCode=None, cmdCode='2709', flowCode='X,M', partnerCode=None,
                                        partner2Code=None, customsCode=None, motCode=None, maxRecords=500, format_output='JSON',
                                        aggregateBy=None, breakdownMode='classic', includeDesc=True)

comtrade_preview.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20220501,2022,5,202205,24,AGO,Angola,M,...,NaN,False,0.0,False,4597.550,4574.63,4597.550,0,False,True
1,C,M,20220501,2022,5,202205,24,AGO,Angola,M,...,NaN,False,0.0,False,4546.310,4546.31,4546.310,0,False,True
2,C,M,20220501,2022,5,202205,24,AGO,Angola,M,...,8.655,False,0.0,False,51.240,28.32,51.240,0,False,True
3,C,M,20220501,2022,5,202205,28,ATG,Antigua and Barbuda,M,...,200.000,False,0.0,False,2395.074,NaN,2395.074,0,False,True
4,C,M,20220501,2022,5,202205,28,ATG,Antigua and Barbuda,M,...,200.000,False,0.0,False,2395.074,NaN,2395.074,0,False,True


In [7]:
comtrade_preview.columns

Index(['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period',
       'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc',
       'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code',
       'partner2ISO', 'partner2Desc', 'classificationCode',
       'classificationSearchCode', 'isOriginalClassification', 'cmdCode',
       'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc',
       'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty',
       'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty',
       'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt',
       'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue',
       'legacyEstimationFlag', 'isReported', 'isAggregate'],
      dtype='object')

In [8]:
# Common params for Crude Oil commodities between 2019-2025
crude_oil_filters = dict(
    typeCode='C', # Commodities
    # freqCode='A', # Annual
    clCode='HS',
    # period='2019,2020,2021,2022,2023,2024,2025',
    reporterCode=None, # No restrictions applied
    cmdCode='2709', # Crude oil
    flowCode='X,M', # Both exports and imports
    partnerCode=None, # No restrictions applied
    partner2Code=None,
    customsCode=None,
    format_output='JSON',
    aggregateBy=None,
    breakdownMode='plus', 
    countOnly=None,
    includeDesc=True # Include country and commodity descriptions
)

In [9]:
# Dataset A: No mode-of-transport restriction
comtrade_all_annual = comtradeapicall.getFinalData(
    subscription_key,
    motCode=None, # Include all modes of transport
    freqCode='A',
    period='2019,2020,2021,2022,2023,2024,2025',
    **crude_oil_filters
)

comtrade_all_annual.columns

Index(['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period',
       'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc',
       'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code',
       'partner2ISO', 'partner2Desc', 'classificationCode',
       'classificationSearchCode', 'isOriginalClassification', 'cmdCode',
       'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc',
       'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty',
       'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty',
       'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt',
       'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue',
       'legacyEstimationFlag', 'isReported', 'isAggregate'],
      dtype='object')

In [10]:
# Dataset A: No mode-of-transport restriction
comtrade_all_monthly = comtradeapicall.getFinalData(
    subscription_key,
    motCode=None, # Include all modes of transport
    freqCode='M',
    period='201905,201906',
    **crude_oil_filters
)

comtrade_all_monthly.columns == comtrade_all_annual.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True])

In [11]:
# Importing monthly-level reports needs 84 API calls
# Uncomment these lines of code if monthly-level report is needed

# import time

# years = range(2019, 2026)
# months = range(1, 13)

# monthly_dfs = []

# for year in years:
#     for month in months:
#         comtrade_monthly = comtradeapicall.getFinalData(
#             subscription_key,
#             motCode=None,
#             freqCode='M',
#             period=f'{year}{month:02d}',
#             **crude_oil_filters
#         )

#         print(f"Downloaded {period}: {len(comtrade_monthly)} rows")
        
#         monthly_dfs.append(comtrade_monthly)

#         time.sleep(1.1)

# comtrade_all_monthly = pd.DataFrame()
# comtrade_all_monthly = pd.concat(monthly_dfs, ignore_index=True)
# comtrade_all_monthly.shape

In [12]:
# Dataset B: Maritime transport only
comtrade_sea_annual = comtradeapicall.getFinalData(
    subscription_key,
    motCode='2100', # Mot code for 'Sea',
    freqCode='A',
    period='2019,2020,2021,2022,2023,2024,2025',
    **crude_oil_filters
)

comtrade_sea_annual.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20190101,2019,52,2019,24,AGO,Angola,M,...,1768.18,False,0.0,False,13738.600,12594.77,13738.600,0,False,True
1,C,A,20190101,2019,52,2019,24,AGO,Angola,M,...,268.18,False,0.0,False,1288.780,1274.31,1288.780,0,False,True
2,C,A,20190101,2019,52,2019,24,AGO,Angola,M,...,1500.00,False,0.0,False,12449.820,11320.46,12449.820,0,False,True
3,C,A,20190101,2019,52,2019,28,ATG,Antigua and Barbuda,M,...,135.60,False,0.0,False,580.737,NaN,580.737,0,False,True
4,C,A,20190101,2019,52,2019,28,ATG,Antigua and Barbuda,M,...,135.60,False,0.0,False,580.737,NaN,580.737,0,False,True


### C. Compare Reporting Countries

Find unique reporting countries and their UN Comtrade country codes in each dataset.

The initial retrieval contains 168 all-mode reporters and 61 sea-mode reporters, indicating more limited country coverage for sea-specific reporting.

In [14]:
all_reporters = set(comtrade_all_annual['reporterDesc'])
sea_reporters = set(comtrade_sea_annual['reporterDesc'])

print("All reporters:", len(all_reporters))
print("Sea reporters:", len(sea_reporters))

All reporters: 168
Sea reporters: 61


In [15]:
annual_reporters_all = (comtrade_all_annual[['reporterCode', 'reporterDesc']].drop_duplicates())
annual_reporters_sea = (comtrade_all_annual[['reporterCode', 'reporterDesc']].drop_duplicates())

In [16]:
annual_reporters_all.head()

,reporterCode,reporterDesc
0,4,Afghanistan
5,20,Andorra
7,24,Angola
10,28,Antigua and Barbuda
12,31,Azerbaijan


In [17]:
annual_reporters_sea.head()

,reporterCode,reporterDesc
0,4,Afghanistan
5,20,Andorra
7,24,Angola
10,28,Antigua and Barbuda
12,31,Azerbaijan


In [18]:
# Reporters with sea only data
sea_only_bool = annual_reporters_sea['reporterCode'].isin(annual_reporters_all['reporterCode'])
annual_reporters_sea.loc[~sea_only_bool]

,reporterCode,reporterDesc


### D. Process Data (Convert Units for Weight)
Convert net weight from kilograms to estimated barrels using:

> barrels = kg ÷ 1,000 × 7.3.

In [20]:
columns = [
    'reporterCode',
    'reporterDesc',
    'partnerCode',
    'partnerDesc',
    'partner2Code',
    'flowCode',
    'flowDesc',
    'motCode',
    'motDesc',
    'period',
    'netWgt',
    'primaryValue'
]

comtrade_all_annual_original = comtrade_all_annual.copy()
comtrade_all_sea_original = comtrade_sea_annual.copy()

comtrade_all_annual = comtrade_all_annual[columns]
comtrade_sea_annual = comtrade_sea_annual[columns]

In [21]:
comtrade_sea_annual

,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,flowCode,flowDesc,motCode,motDesc,period,netWgt,primaryValue
0,24,Angola,0,World,0,M,Import,2100,Sea,2019,1.768180e+03,1.373860e+04
1,24,Angola,724,Spain,0,M,Import,2100,Sea,2019,2.681800e+02,1.288780e+03
2,24,Angola,826,United Kingdom,0,M,Import,2100,Sea,2019,1.500000e+03,1.244982e+04
3,28,Antigua and Barbuda,0,World,0,M,Import,2100,Sea,2019,1.356000e+02,5.807370e+02
4,28,Antigua and Barbuda,842,USA,0,M,Import,2100,Sea,2019,1.356000e+02,5.807370e+02
...,...,...,...,...,...,...,...,...,...,...,...,...
8538,724,Spain,380,Italy,380,X,Export,2100,Sea,2025,7.664662e+07,5.455383e+07
8539,724,Spain,0,World,380,X,Export,2100,Sea,2025,7.664662e+07,5.455383e+07
8540,724,Spain,380,Italy,380,X,Export,2100,Sea,2025,7.664662e+07,5.455383e+07
8541,826,United Kingdom,0,World,380,X,Export,2100,Sea,2025,3.510845e+08,1.760186e+08


In [22]:
comtrade_all_annual['barrels'] =  (comtrade_all_annual['netWgt'] / 1000) * 7.3
comtrade_sea_annual['barrels'] =  (comtrade_sea_annual['netWgt'] / 1000) * 7.3